执行命令需要在同一行命令中，先 source 环境名（base、modelone 等）才能 pip 安装到指定环境；如果不知道有哪些虚拟环境，可以运行 conda info --envs 查看

In [ ]:
!source activate modelone && pip install pandas

In [ ]:
from cubestudio.request.model_client import Client,init
from cubestudio.train.task import Job_Template,Project,Pipeline,Task
import json

In [ ]:
# 初始化客户端
import os
HOST = os.environ['MODELONE_API_URL']
token = os.environ['MODELONE_API_TOKEN']
username = os.environ.get('MODELONE_USERNAME', 'admin')
init(host=HOST,username=username,token=token)

In [ ]:
# 添加一个画布
pipeline = Client(Pipeline).add_or_update(
    name=f'{username}-default',
    describe='sdk画布',
    project=Client(Project).one(name='public')
)

In [ ]:
# 添加第1个任务
job_template = Client(Job_Template).one(name="自定义镜像")
task=Client(Task).add_or_update(
    name='sdk-test1',
    label='sdk发起的任务1',
    pipeline=pipeline,
    job_template=Client(Job_Template).one(name="自定义镜像"),
    timeout=3600,
    retry=0,
    args=json.dumps(
        {
            "images":"ubuntu:20.04",
            "command":'for i in {1..50}; do date; sleep 1; done',
            "workdir":"/"
        }
    )
)

In [ ]:
# 添加第2个任务
job_template = Client(Job_Template).one(name="自定义镜像")
task=Client(Task).add_or_update(
    name='sdk-test2',
    label='sdk发起的任务2',
    pipeline=pipeline,
    job_template=Client(Job_Template).one(name="自定义镜像"),
    timeout=3600,
    retry=0,
    args=json.dumps(
        {
            "images":"ubuntu:20.04",
            "command":'for i in {1..50}; do date; sleep 1; done',
            "workdir":"/"
        }
    )
)

In [ ]:
# 设置节点上下游关系
pipeline.update(dag_json=json.dumps(
    {
        "sdk-test2": {
            "upstream": ["sdk-test1"]
        }
    }
))

In [ ]:
pipeline.run()